# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly
# only build with ONNX gpu + caspar + ceres cuda/cudss; downloads disabled, so model paths must be set
!pip uninstall -y --quiet pycolmap
!pip install --progress-bar off --quiet \
   "https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl"

In [ ]:
from tqdm import tqdm
from pathlib import Path
import hashlib
import json
import numpy as np
import os
import shutil
import subprocess
import urllib.request


from hloc import reconstruction, visualization
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d
import pycolmap

In [ ]:
DATA_PATH = next(p for p in [Path('/kaggle/input/datasets/yu5uf5/buggy-hloc'),
                             Path('/kaggle/input/buggy-hloc')] if p.exists())
outputs = Path('/kaggle/working/multi')
outputs.mkdir(exist_ok=True)
IMAGES_PATH = Path('/tmp/images')

In [ ]:
# camera position relative to the racebox in body frame, meters: {roll_id: (forward, left)}
# +forward = camera ahead of the racebox, +left = camera to its left
RB_CAM_OFFSET = {
    37: (1.25, 0), 38: (1.25, 0),
    39: (1.20, 0),
    44: (-0.80, 0), 45: (-0.80, 0),
    1387: (0.05, 0),
    1388: (-0.75, 0),
    1401: (-0.70, 0)
}

R_EARTH = 6371000.0

def apply_rb_cam_offset(gps, fwd, left):
    """Move racebox positions to the camera using gps-derived heading."""
    lat = np.array([s['lat'] for s in gps], float)
    lon = np.array([s['long'] for s in gps], float)
    latr = np.radians(lat)
    i = np.arange(len(gps))
    i0, i1 = np.maximum(i - 12, 0), np.minimum(i + 12, len(gps) - 1)  # ~0.5 s window at 25 Hz
    dn = np.radians(lat[i1] - lat[i0]) * R_EARTH
    de = np.radians(lon[i1] - lon[i0]) * R_EARTH * np.cos(latr)
    ok = np.hypot(de, dn) > 0.5
    if not ok.any():
        return
    head = np.arctan2(de, dn)
    last = np.maximum.accumulate(np.where(ok, i, -1))
    last[last < 0] = np.flatnonzero(ok)[0]  # hold heading through standstill
    head = head[last]
    off_e = fwd * np.sin(head) - left * np.cos(head)
    off_n = fwd * np.cos(head) + left * np.sin(head)
    for s, oe, on, phi in zip(gps, off_e, off_n, latr):
        s['lat'] += np.degrees(on / R_EARTH)
        s['long'] += np.degrees(oe / (R_EARTH * np.cos(phi)))

In [ ]:
import cv2
from collections import defaultdict
from itertools import combinations
from scipy.spatial import cKDTree


def load_vid_imu(vid_imu_path):
    """Load vid_imu exports; shift racebox onto the (offset-corrected) frame timeline
    and move it to the camera."""
    data = {}
    for p in sorted(vid_imu_path.glob('*.json')):
        with open(p) as f:
            data[p.stem] = json.load(f)
    for run, d in data.items():
        # racebox offset is imu-estimated in smooth.ipynb's export; TIME_OFFSETS holds the
        # measured residual frame-time error (shift gps by -dt == interp at frame_ts + dt)
        off_ns = (d.get('racebox_offset_ms', 0.0) - TIME_OFFSETS.get(run, 0.0)) * 1e6
        for key in ('racebox_gps', 'racebox_speed'):
            for s in d.get(key, []):
                s['timestamp'] += off_ns
        fwd, left = RB_CAM_OFFSET.get(int(run), (0.0, 0.0))
        if fwd or left:
            apply_rb_cam_offset(d.get('racebox_gps', []), fwd, left)
    return data


def sharpness_score(bgr):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def extract_frames(video, start_ns, out_dir, stride=1):
    """Save frames as <ts_ns>.jpg plus sharpness.json; skips folders already done."""
    if (out_dir / 'sharpness.json').exists():
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")
    scores = {}
    i = 0
    try:
        with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc=out_dir.name, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % stride != 0:
                    continue
                ts_ns = start_ns + int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1_000_000))
                cv2.imwrite(str(out_dir / f"{ts_ns}.jpg"), frame)
                scores[ts_ns] = sharpness_score(frame)
    finally:
        cap.release()
    with open(out_dir / 'sharpness.json', 'w') as f:
        json.dump(scores, f)
    print(f"{out_dir.name}: saved {len(scores)} frames")


def gps_enu(d, field):
    gps = d[field]
    ts, lat, lon, alt = (np.array([s[k] for s in gps], float)
                         for k in ('timestamp', 'lat', 'long', 'alt'))
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([lat, lon, alt], 1)), LAT0, LON0, ALT0))
    enu[:, 2] += CAM_HEIGHT
    return ts, enu


def speed_arrays(d, kind):
    if kind == 'racebox':
        spd = d['racebox_speed']
        return (np.array([s['timestamp'] for s in spd], float),
                np.array([s['speed'] for s in spd], float))
    vel = d['velocity']
    return (np.array([s['timestamp'] for s in vel], float),
            np.hypot([s['vx'] for s in vel], [s['vy'] for s in vel]))


def select_frames(run, d, gps_kind, delta_s):
    """Distance-uniform selection over the video ∩ gps window.
    Returns ({names, enu, heading}, (gps_ts, enu_src))."""
    image_paths = sorted((IMAGES_PATH / run).glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    gps_ts, enu_src = gps_enu(d, 'racebox_gps' if gps_kind == 'racebox' else 'gps_data')
    spd_ts, spd = speed_arrays(d, gps_kind)

    m = (spd_ts >= max(gps_ts[0], image_ts[0])) & (spd_ts <= min(gps_ts[-1], image_ts[-1]))
    ts_w, v_w = spd_ts[m], spd[m]
    dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
    sel_ts = np.interp(np.arange(0.0, dist[-1], delta_s), dist, ts_w)

    # sharpest frame within each target's window (blur is vibration-driven, varies frame to frame)
    scores = {}
    score_file = IMAGES_PATH / run / 'sharpness.json'
    if score_file.exists():
        with open(score_file) as f:
            scores = {int(k): v for k, v in json.load(f).items()}
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2],
                             (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
        elif scores:
            idx.append(i0 + int(np.argmax([scores.get(int(u), 0.0) for u in image_ts[i0:i1]])))
        else:
            idx.append(i0 + np.abs(image_ts[i0:i1] - t).argmin())
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, gps_ts, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    print(f'{run}: {len(idx)} frames over {dist[-1]:.0f}m')
    return ({'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
             'enu': enu,
             'heading': np.arctan2(grad[:, 1], grad[:, 0])},
            (gps_ts, enu_src))


def sequential_pairs(names, k):
    return {(names[i], names[j])
            for i in range(len(names))
            for j in range(i + 1, min(i + 1 + k, len(names)))}


def cross_pairs(sel_a, sel_b, k, r, heading_max_deg):
    """Proximity pairs between two selections, gated on heading difference."""
    tree = cKDTree(sel_b['enu'][:, :2])
    dists, nbrs = tree.query(sel_a['enu'][:, :2], k=k, distance_upper_bound=r)
    if k == 1:
        dists, nbrs = dists[:, None], nbrs[:, None]
    pairs = set()
    for i, (ds, js) in enumerate(zip(dists, nbrs)):
        for dist, j in zip(ds, js):
            if not np.isfinite(dist):
                continue
            dh = abs(sel_a['heading'][i] - sel_b['heading'][j])
            if min(dh, 2 * np.pi - dh) <= np.radians(heading_max_deg):
                pairs.add(tuple(sorted((sel_a['names'][i], sel_b['names'][j]))))
    return pairs

In [ ]:
MODELS_PATH = Path('/tmp/models')
MASKS_PATH = Path('/tmp/masks')
LG_FP16 = True     # fp16 matcher: 1.8x pairs/s on a T4, poses agree to 4 cm p90
LG_FP16_CACHE = os.environ.get('SRS_LG_FP16_CACHE')   # optional dir or rclone target to stage from


def lightglue_model():
    """Matcher weights: fp16, converted from the fp32 asset in seconds; fp32 if LG_FP16 off."""
    f32 = MODELS_PATH / 'aliked-lightglue.onnx'
    if not LG_FP16:
        return f32
    f16 = MODELS_PATH / 'aliked-lightglue-fp16.onnx'
    if f16.exists():
        return f16
    if LG_FP16_CACHE:
        src = f'{LG_FP16_CACHE.rstrip("/")}/{f16.name}'
        if Path(src).exists():
            shutil.copy(src, f16)
        elif shutil.which('rclone'):
            subprocess.run(['rclone', 'copyto', src, str(f16)])
    if not f16.exists():
        import onnx
        # onnxconverter_common.float16 1.16.0 crashes on this graph; onnxruntime's works
        from onnxruntime.transformers.float16 import convert_float_to_float16
        onnx.save(convert_float_to_float16(onnx.load(f32), keep_io_types=True,
                                           disable_shape_infer=True), f16)
    # the measured fp16 accuracy is one conversion's; make a library change visible
    print(f'{f16.name}: sha256 {hashlib.sha256(f16.read_bytes()).hexdigest()}')
    return f16


def stage_models():
    # this pycolmap build can't auto-download onnx models; stage them locally
    MODELS_PATH.mkdir(exist_ok=True)
    for name in ('aliked-n16rot.onnx', 'aliked-lightglue.onnx'):
        f = MODELS_PATH / name
        if not f.exists():
            urllib.request.urlretrieve(
                f'https://github.com/colmap/colmap/releases/download/3.13.0/{name}', f)
    lightglue_model()


def link_masks(names):
    # colmap masks: <mask_path>/<image name>.png, black = exclude; all frames share mask0
    MASKS_PATH.mkdir(exist_ok=True)
    mask_src = MASKS_PATH / 'mask0.png'
    if not mask_src.exists():
        shutil.copy(DATA_PATH / 'mask0.png', mask_src)
    for ref in names:
        dst = MASKS_PATH / f'{ref}.png'
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.link(mask_src, dst)


def extract_image_features(db_path, names, camera_params=None, masks=True):
    options = pycolmap.FeatureExtractionOptions()
    options.type = pycolmap.FeatureExtractorType.ALIKED_N16ROT
    options.aliked.n16rot_model_path = str(MODELS_PATH / 'aliked-n16rot.onnx')
    options.aliked.max_num_features = 4096
    options.use_gpu = True
    options.gpu_index = GPU_INDEX
    reader = {'camera_model': 'SIMPLE_RADIAL',
              'camera_params': ','.join(str(v) for v in (camera_params or CAMERA_PARAMS))}
    if masks:
        reader['mask_path'] = str(MASKS_PATH)
    pycolmap.extract_features(
        db_path, IMAGES_PATH, image_names=sorted(names),
        camera_mode=pycolmap.CameraMode.PER_FOLDER,  # consider PER_IMAGE because of stabilization
        reader_options=reader,
        extraction_options=options)


def match_pairs(db_path, pairs_file):
    matching_options = pycolmap.FeatureMatchingOptions()
    matching_options.type = pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE
    matching_options.aliked.lightglue.model_path = str(lightglue_model())
    matching_options.use_gpu = True
    matching_options.gpu_index = GPU_INDEX
    pairing_options = pycolmap.ImportedPairingOptions()
    pairing_options.match_list_path = str(pairs_file)
    pycolmap.match_image_pairs(db_path, matching_options=matching_options,
                               pairing_options=pairing_options)


def write_pose_priors(db_path, gps_by_run, cov):
    """Position priors for images of runs in gps_by_run; inert unless use_prior_position."""
    with pycolmap.Database.open(str(db_path)) as db:
        have = {p.corr_data_id.id for p in db.read_all_pose_priors()}
        for image in db.read_all_images():
            p = Path(image.name)
            if image.data_id.id in have or p.parts[0] not in gps_by_run:
                continue
            gps_ts, enu = gps_by_run[p.parts[0]]
            ts = int(p.stem)
            prior = pycolmap.PosePrior()
            prior.corr_data_id = image.data_id
            prior.position = np.array([np.interp(ts, gps_ts, enu[:, i]) for i in range(3)])
            prior.position_covariance = cov
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
        print('pose priors:', db.num_pose_priors())

In [ ]:
videos = {v.stem: v for v in DATA_PATH.glob('*.mp4')}
data = load_vid_imu(DATA_PATH / 'vid_imu')

# Create Images

In [ ]:
for stem, video in tqdm(videos.items(), desc='videos'):
    extract_frames(video, data[stem]['camera_start'], IMAGES_PATH / stem)

# Config

In [ ]:
sfm_pairs = outputs / 'pairs-sfm.txt'
sfm_dir = outputs / 'sfm'
sfm_prior_dir = outputs / 'sfm_prior'
database = outputs / 'database.db'

RESUME = False     # continue from outputs/database.db + outputs/prior
RECON_RUNS = None  # runs to reconstruct this pass; None = all

In [ ]:
# racebox: 25Hz + scalar doppler speed; fit: 10Hz + velocity vectors
GPS_KIND = 'racebox'
LAT0, LON0, ALT0 = 40.44163016, -79.94165829, 288.42151354  # shared ENU reference
gt = pycolmap.GPSTransform(pycolmap.GPSTransfromEllipsoid.WGS84)

DELTA_S = 2.0         # m between selected frames
SEQ_K = 6             # forward sequential pairs per frame
CROSS_K = 3           # nearest cross-run candidates per frame
CROSS_R = 6.0         # m, max cross-run pair distance
HEADING_MAX_DEG = 40  # max cross-run heading difference

CAM_HEIGHT = 0.35  # m, gps alt is DEM road level; lift priors approximatley to the camera

# measured frame-time offsets, ms (dt grid-fit of model/localized poses vs racebox):
# gps interp at frame_ts + dt fits best; load_vid_imu applies it by shifting the gps streams
TIME_OFFSETS = {'37': -60.0, '38': -70.0, '45': -60.0, '1388': 60.0, '1401': 0.0,
                '39': -50.0, '44': -50.0, '1387': -1300.0}

In [ ]:
import torch

GPU_INDEX = ','.join(str(i) for i in range(torch.cuda.device_count()))

# caspar BA only supports SIMPLE_RADIAL [f, cx, cy, k1]/PINHOLE
# initial estimates refined per image
CAMERA_PARAMS = [653.4, 631.72, 338.74, -0.0526]

# Mapping

## Select Frames

In [ ]:
runs = sorted(data, key=int)
sel, run_enu = {}, {}
for run in runs:
    sel[run], run_enu[run] = select_frames(run, data[run], GPS_KIND, DELTA_S)

# selection must reproduce db names (guards param drift across sessions)
if RESUME:
    with pycolmap.Database.open(str(database)) as db:
        db_names = {im.name for im in db.read_all_images()}
    for run in runs:
        mine = {n for n in db_names if n.startswith(f'{run}/')}
        assert not mine or mine == set(sel[run]['names']), run

references = [n for run in runs for n in sel[run]['names']]
recon_names = sorted(n for r in (RECON_RUNS or runs) for n in sel[r]['names'])
len(references)

In [ ]:
import matplotlib.pyplot as plt

for run in runs:
    plt.plot(*sel[run]['enu'][:, :2].T, '.', ms=2, label=run)
plt.axis('equal')
plt.legend(markerscale=5)
plt.show()

In [ ]:
sl = slice(200, 210)
plot_images([read_image(IMAGES_PATH / ref) for ref in references[sl]],
            titles=references[sl], dpi=25)

## Features

In [ ]:
stage_models()
link_masks(references)

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, outputs / 'colmap.LOG.')
if not RESUME:
    database.unlink(missing_ok=True)
extract_image_features(database, references)

## Matching

In [ ]:
pairs = set()
for run in runs:
    pairs |= sequential_pairs(sel[run]['names'], SEQ_K)
n_seq = len(pairs)

for a, b in combinations(runs, 2):
    pairs |= cross_pairs(sel[a], sel[b], CROSS_K, CROSS_R, HEADING_MAX_DEG)

sfm_pairs.write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(f'{n_seq} sequential + {len(pairs) - n_seq} cross-run pairs')

In [ ]:
match_pairs(database, sfm_pairs)

## Reconstruct

### Database

In [ ]:
PRIOR_STD_XY = 0.5
PRIOR_STD_Z = 1.0
# inert unless use_prior_position is set, so both reconstructions can share the db
write_pose_priors(database, run_enu,
                  np.diag([PRIOR_STD_XY**2, PRIOR_STD_XY**2, PRIOR_STD_Z**2]))

### No Prior

In [ ]:
# model = reconstruction.run_reconstruction(
#     sfm_dir, database, IMAGES_PATH, verbose=True,
#     options={
#         "image_names": recon_names,
#         "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#         "ba_global_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#     })

In [ ]:
# p = outputs / 'no_prior'
# p.mkdir(parents=True, exist_ok=True)
# model.write(p)

### Prior

In [ ]:
# priors only constrain global BA, and caspar doesn't support them
prior_options = {
    "image_names": recon_names,
    "use_prior_position": True,
    # "use_robust_loss_on_prior_position": True,
    # Speed options
    # "ba_use_gpu": True,
    # "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
    # "ba_global_backend": pycolmap.BundleAdjustmentBackend.CERES,
    # "ba_global_frames_ratio": 1.3,
    # "ba_global_points_ratio": 1.3,
    # "ba_local_max_num_iterations": 12,
    # "ba_local_max_refinements": 2,
    # "ba_global_max_num_iterations": 30,
    # "ba_global_max_refinements": 3,
    # "mapper": {"ba_global_ignore_redundant_points3D": True},
}
if RESUME:
    shutil.rmtree(sfm_prior_dir, ignore_errors=True)
    sfm_prior_dir.mkdir(parents=True)
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(
            database, IMAGES_PATH, sfm_prior_dir,
            options={**prior_options, "fix_existing_frames": False},
            input_path=str(outputs / 'prior'))
    model_prior = recs[0]
else:
    model_prior = reconstruction.run_reconstruction(
        sfm_prior_dir, database, IMAGES_PATH, verbose=True, options=prior_options)

In [ ]:
p = outputs / 'prior'
p.mkdir(parents=True, exist_ok=True)
model_prior.write(p)

# RTK Anchoring

Anchor the map to robobuggy RTK — final validated recipe: robo left-eye frames register against the VIRB map, carry per-anchor EKF covariances (horizontal **and** vertical), then a smooth prewarp + full retriangulation jumps the map into the prior-consistent basin before a converged joint BA.

Validated: held-out info-weighted **0.29 m** (median anchor ≈1.2σ of its RTK claim, the nominal value); road z within ±0.1 m of the USGS LiDAR DEM. Tested & rejected: RS rectification, CLAHE, stereo rigs (rights pruned — `apply_rig_config` renumbers frames and desyncs any db resumed against an existing model), anisotropic along-track covariance (the ~50 ms in-bag stamp jitter is real but modeling it per-anchor discards useful signal), robust prior loss during mapping (saturates while the model is far → priors ignored). Reprojection errors are correlated (vibration/RS/blur), so priors run at σ/`PRIOR_SCALE` — the scale was validated on held-out anchors, not fitted to training residuals. Raw z-rmse vs RTK is misleading (per-day RTK z biases): judge vertical accuracy against the DEM.

In [ ]:
ROBO_PATH = next(p for p in [Path('/kaggle/input/datasets/yu5uf5/buggy-robo'),
                             Path('/kaggle/input/buggy-robo')] if p.exists())
rtk_work = Path('/tmp/rtk')
rtk_out = Path('/kaggle/working/rtk')

CAM_FROM_GNSS2 = np.array([0.05, 0.0, -0.01])  # m, body frame; camera a few cm ahead of the antenna
ROBO_TIME_OFFSET_MS = -8.0     # nominal mid-readout+exposure lead for runs without a measured dt
# measured per-run offsets (dt grid-fit; ZED verified independently by paint-line crossings,
# and includes the directly-measured 25 ms H.264 decoder reorder lag)
TIME_OFFSETS.update({'2026-03-21_sc_0822': 60.0, '2025-10-04_4_sc_lgap': -25.0,
                     '2025-10-04_2_sc_whobaat': 0.0})
ROBO_PRIOR_STD_FLOOR = 0.08    # m, horizontal
ROBO_READOUT_S = 0.010         # rolling-shutter prior inflation: var += (v*t/2)^2
ROBO_PRIOR_STD_Z_FLOOR = 0.08  # m; z uses the per-anchor EKF variance, never a flat sigma
VIRB_PRIOR_STD_XY, VIRB_PRIOR_STD_Z = 3.0, 2.0  # loosened; RTK is the georeference
PRIOR_SCALE = 8.0  # solver-side sigma/8 on robo priors: correlated reprojection noise
                   # overweights vision. Swept 2/3/5/8 with held-out gating (held-out
                   # info-weighted 0.32/0.29/0.26/0.23) AND truth-sensitive checks:
                   # sigma/8 improved field smoothness, cross-day relative consistency,
                   # and the LiDAR-DEM z profile — i.e. its gains are real, not fitted

In [ ]:
robo = {}
for pf in sorted(ROBO_PATH.glob('*/poses.json')):
    with open(pf) as f:
        robo[pf.parent.name] = {'dir': pf.parent, **json.load(f)}


def quat_rotate(q, v):
    # q (n,4) xyzw, v (3,) -> (n,3)
    qv, w = q[:, :3], q[:, 3:]
    t = 2 * np.cross(qv, v)
    return v + w * t + np.cross(qv, t)


def ecef_to_enu_rot(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    sla, cla, slo, clo = np.sin(la), np.cos(la), np.sin(lo), np.cos(lo)
    return np.array([[-slo, clo, 0.0],
                     [-sla * clo, -sla * slo, cla],
                     [cla * clo, cla * slo, sla]])


def robo_cam_enu(run):
    """(ts_ns on the frame timeline, camera enu, horizontal var, vertical var, speed)."""
    r = robo[run]
    e = r['ekf']
    dt_ms = TIME_OFFSETS.get(run, ROBO_TIME_OFFSET_MS)
    ts = (np.array(e['t'], float) * 1000 - dt_ms) * 1e6
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([e['lat'], e['lon'], e['alt']], 1)),
                                       LAT0, LON0, ALT0))
    # ekf pose is of imu_link; move to the camera with the body lever arm
    lever = np.array(r['tf']['imu_link->gnss_2_antenna_link']) + CAM_FROM_GNSS2
    enu += quat_rotate(np.array(e['quat']), lever) @ ecef_to_enu_rot(LAT0, LON0).T
    from scipy.ndimage import maximum_filter1d
    # rolling max so brief covariance spikes aren't understated between samples
    var = maximum_filter1d(np.array(e['pos_var'])[:, :2].mean(1), size=5)
    var = np.maximum(var, ROBO_PRIOR_STD_FLOOR**2)
    var_z = maximum_filter1d(np.array(e['pos_var'])[:, 2], size=5)
    var_z = np.maximum(var_z, ROBO_PRIOR_STD_Z_FLOOR**2)
    spd = np.array(e['speed'])
    var = var + (spd * ROBO_READOUT_S / 2)**2
    return ts, enu, var, var_z, spd

def robo_roll_windows(run, v_on=1.2, bridge_s=8.0, pad_s=3.0, min_seg_m=50.0):
    """[(t0_ns, t1_ns)] moving segments of the roll — drops staging / parked / walk-back.
    Multiple windows survive (e.g. out-and-back passes); tiny segments don't."""
    e = robo[run]['ekf']
    t = np.array(e['t'], float)
    v = np.array(e['speed'], float)
    k = max(1, int(2 / np.median(np.diff(t))))
    vs = np.convolve(v, np.ones(k) / k, 'same')
    idx = np.flatnonzero(vs > v_on)
    segs = np.split(idx, np.flatnonzero(np.diff(t[idx]) > bridge_s) + 1)
    dist = np.concatenate([[0], np.cumsum(np.diff(t) * (v[1:] + v[:-1]) / 2)])
    return [((t[s[0]] - pad_s) * 1e9, (t[s[-1]] + pad_s) * 1e9)
            for s in segs if dist[s[-1]] - dist[s[0]] >= min_seg_m]


print(sorted(robo))

In [ ]:
import av
import base64
from mcap.reader import make_reader


def export_svo_frames(run):
    r = robo[run]
    out = {s: IMAGES_PATH / run / s for s in ('left', 'right')}
    for d_ in out.values():
        d_.mkdir(parents=True, exist_ok=True)
    off_ns = int(round(r['sync']['offset_s'] * 1e9))
    t0, t1 = int(r['ekf']['t'][0] * 1e9), int(r['ekf']['t'][-1] * 1e9)
    codec = av.CodecContext.create('h264', 'r')
    scores, zed_ts = {}, {}
    gyro_t, gyro_w = [], []
    with open(r['dir'] / 'vid.svo2', 'rb') as f:
        reader = make_reader(f)
        with tqdm(total=r['camera']['num_frames'], desc=run, leave=False) as progress:
            for _, ch, msg in reader.iter_messages():
                if ch.topic.endswith('/sensors'):
                    buf = base64.b64decode(json.loads(msg.data)['data'])
                    gyro_t.append(int.from_bytes(buf[16:24], 'little'))
                    gyro_w.append(float(np.linalg.norm(np.frombuffer(buf[88:100], '<f4'))))
                    continue
                if not ch.topic.endswith('/side_by_side'):
                    continue
                progress.update(1)
                try:
                    frames = [fr for pkt in codec.parse(bytes(msg.data[8:]))
                              for fr in codec.decode(pkt)]
                except av.error.InvalidDataError:  # corrupt message; decode recovers at next keyframe
                    codec = av.CodecContext.create('h264', 'r')
                    continue
                for fr in frames:
                    ts_ns = msg.log_time + off_ns
                    if not (t0 <= ts_ns <= t1):
                        continue
                    img = fr.to_ndarray(format='bgr24')
                    w = img.shape[1] // 2
                    cv2.imwrite(str(out['left'] / f'{ts_ns}.jpg'), img[:, :w])
                    cv2.imwrite(str(out['right'] / f'{ts_ns}.jpg'), img[:, w:])
                    scores[ts_ns] = sharpness_score(img[:, :w])
                    zed_ts[ts_ns] = msg.log_time
    gyro_t = np.array(gyro_t, float)
    gyro_w = np.array(gyro_w)
    omega = {ts: float(gyro_w[m].mean()) if (m := np.abs(gyro_t - zt) < 15e6).any() else 0.0
             for ts, zt in zed_ts.items()}
    return scores, omega


def export_bag_frames(run):
    """In-bag jpeg camera: R/B channels are swapped and rare frames carry burned-in overlays."""
    from rosbags.highlevel import AnyReader
    r = robo[run]
    out = IMAGES_PATH / run / 'left'
    out.mkdir(parents=True, exist_ok=True)
    t0, t1 = int(r['ekf']['t'][0] * 1e9), int(r['ekf']['t'][-1] * 1e9)
    scores = {}
    imu_t, imu_w = [], []
    dropped = 0
    with AnyReader([r['dir'] / 'data.mcap']) as reader:
        conns = [c for c in reader.connections if c.topic in (r['camera']['topic'], '/imu/data')]
        for conn, ts, raw in tqdm(reader.messages(connections=conns), desc=run, leave=False):
            m = reader.deserialize(raw, conn.msgtype)
            if conn.topic == '/imu/data':
                w = m.angular_velocity
                imu_t.append(ts)
                imu_w.append(np.degrees((w.x**2 + w.y**2 + w.z**2) ** 0.5))
                continue
            if not (t0 <= ts <= t1):
                continue
            img = cv2.imdecode(np.frombuffer(m.data, np.uint8), cv2.IMREAD_COLOR)[:, :, ::-1]
            b, g, rr = img[:, :, 0].astype(int), img[:, :, 1].astype(int), img[:, :, 2].astype(int)
            if ((np.maximum(np.maximum(b, g), rr) >= 250) &
                    (np.minimum(np.minimum(b, g), rr) <= 60)).sum() > 500:
                dropped += 1
                continue
            cv2.imwrite(str(out / f'{ts}.jpg'), img)
            scores[ts] = sharpness_score(img)
    imu_t = np.array(imu_t, float)
    imu_w = np.array(imu_w)
    omega = {ts: float(imu_w[m].mean()) if (m := np.abs(imu_t - ts) < 60e6).any() else 0.0
             for ts in scores}
    print(f'{run}: dropped {dropped} overlay frames')
    return scores, omega


def export_robo_frames(run):
    out_l = IMAGES_PATH / run / 'left'
    if (out_l / 'sharpness.json').exists():
        return
    if robo[run]['camera'].get('source', 'svo2') == 'bag_jpeg':
        scores, omega = export_bag_frames(run)
    else:
        scores, omega = export_svo_frames(run)
    with open(out_l / 'sharpness.json', 'w') as f:
        json.dump(scores, f)
    with open(out_l / 'omega.json', 'w') as f:
        json.dump(omega, f)
    print(f'{run}: saved {len(scores)} frames')


for run in robo:
    export_robo_frames(run)

In [ ]:
def select_robo_frames(run, delta_s):
    """Distance-uniform windows; sharp-enough frame with the calmest gyro per window."""
    img_dir = IMAGES_PATH / run / 'left'
    image_paths = sorted(img_dir.glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    ts_p, enu_src, var, _, spd = robo_cam_enu(run)
    sel_ts = []
    for w0, w1 in robo_roll_windows(run):
        m = (ts_p >= max(image_ts[0], w0)) & (ts_p <= min(image_ts[-1], w1))
        if m.sum() < 2:
            continue
        ts_w, v_w = ts_p[m], spd[m]
        dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
        sel_ts.extend(np.interp(np.arange(0.0, dist[-1], delta_s), dist, ts_w))
    sel_ts = np.array(sel_ts)

    with open(img_dir / 'sharpness.json') as f:
        scores = {int(k): v for k, v in json.load(f).items()}
    with open(img_dir / 'omega.json') as f:
        omega = {int(k): v for k, v in json.load(f).items()}
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2], (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
            continue
        cand = image_ts[i0:i1]
        sh = np.array([scores.get(int(u), 0.0) for u in cand])
        om = np.array([omega.get(int(u), np.inf) for u in cand])
        vv = np.interp(cand, ts_p, var)
        ok = sh >= np.percentile(sh, 60)
        ok &= vv <= 4 * vv.min()  # covariance spikes lose to much-cleaner neighbors
        if ok.any():
            om[~ok] = np.inf
            idx.append(i0 + int(np.argmin(om)))
        else:
            idx.append(i0 + int(np.argmax(sh)))
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, ts_p, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    print(f'{run}: {len(idx)} frames selected')
    return {'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
            'ts': ts, 'enu': enu, 'heading': np.arctan2(grad[:, 1], grad[:, 0])}


robo_sel = {run: select_robo_frames(run, DELTA_S) for run in robo}

# map frame positions/headings from the model; drop robo frames facing against the course
rtk_map_model = DATA_PATH / 'prior_all'
map_model = pycolmap.Reconstruction(str(rtk_map_model))
by_run = defaultdict(list)
for i in map_model.reg_image_ids():
    im = map_model.images[i]
    p = Path(im.name)
    by_run[p.parts[0]].append((int(p.stem), im.name, im.projection_center()))
map_sel = {}
for run, items in by_run.items():
    items.sort()
    enu = np.array([c for _, _, c in items])
    grad = np.gradient(enu[:, :2], axis=0)
    map_sel[run] = {'names': [n for _, n, _ in items], 'enu': enu,
                    'heading': np.arctan2(grad[:, 1], grad[:, 0])}

mm_enu = np.vstack([m['enu'][:, :2] for m in map_sel.values()])
mm_head = np.concatenate([m['heading'] for m in map_sel.values()])
tree = cKDTree(mm_enu)
for run, s in robo_sel.items():
    d, j = tree.query(s['enu'][:, :2], k=1)
    dh = np.abs(mm_head[j] - s['heading'])
    dh = np.minimum(dh, 2 * np.pi - dh)
    ok = (d < 8.0) & (dh < np.pi / 2)
    robo_sel[run] = {'names': [n for n, o in zip(s['names'], ok) if o],
                     'ts': s['ts'][ok], 'enu': s['enu'][ok], 'heading': s['heading'][ok]}
    print(f'{run}: kept {ok.sum()}/{len(ok)} course-direction frames')

robo_names = {run: {'left': s['names']} for run, s in robo_sel.items()}
# left eyes only: stereo rigs were tested and dropped (see section notes)

In [ ]:
rtk_work.mkdir(parents=True, exist_ok=True)
rtk_db = rtk_work / 'database.db'
if not rtk_db.exists():
    shutil.copy(DATA_PATH / 'database.db', rtk_db)
stage_models()


def zed_conf(run, section):
    cur, vals = None, {}
    for line in robo[run]['camera']['factory_calibration_conf'].splitlines():
        line = line.strip()
        if line.startswith('['):
            cur = line.strip('[]')
        elif '=' in line and cur == section:
            k, v = line.split('=')
            vals[k] = float(v)
    return vals


BAG_CAM_SEED = [1088.0, 640.0, 360.0, 0.0]  # unknown intrinsics; refined during phase 1

for run in robo_sel:
    if robo[run]['camera'].get('source', 'svo2') == 'bag_jpeg':
        extract_image_features(rtk_db, robo_names[run]['left'], masks=False,
                               camera_params=BAG_CAM_SEED)
    else:
        v = zed_conf(run, 'LEFT_CAM_HD')
        extract_image_features(rtk_db, robo_names[run]['left'], masks=False,
                               camera_params=[(v['fx'] + v['fy']) / 2, v['cx'], v['cy'], v['k1']])

In [ ]:
import sqlite3

# replace the mapping-era racebox priors: loose racebox on VIRB frames, tight RTK on robo left
con = sqlite3.connect(rtk_db)
con.execute('DELETE FROM pose_priors')
con.commit()
con.close()

data = load_vid_imu(DATA_PATH / 'vid_imu')
virb_enu = {run: gps_enu(d, 'racebox_gps') for run, d in data.items()}
write_pose_priors(rtk_db, virb_enu,
                  np.diag([VIRB_PRIOR_STD_XY**2, VIRB_PRIOR_STD_XY**2, VIRB_PRIOR_STD_Z**2]))

# optional drop list: torn frames (split-motion tear detector, > 30 px) and turnaround
# tails simply get no prior — their features still contribute, their positions don't
DROP_NAMES = set()
tear_file = rtk_work / 'tear_scan.json'
if tear_file.exists():
    with open(tear_file) as f:
        DROP_NAMES = {n for n, v in json.load(f).items() if v['split'] > 30}

with pycolmap.Database.open(str(rtk_db)) as db:
    name_to_img = {im.name: im for im in db.read_all_images()}
    n = 0
    for run, s in robo_sel.items():
        ts_p, enu_src, var, var_z, _ = robo_cam_enu(run)
        for name, t in zip(s['names'], s['ts']):
            if name in DROP_NAMES:
                continue
            prior = pycolmap.PosePrior()
            prior.corr_data_id = name_to_img[name].data_id
            prior.position = np.array([np.interp(t, ts_p, enu_src[:, i]) for i in range(3)])
            v = float(np.interp(t, ts_p, var))
            vz = float(np.interp(t, ts_p, var_z))
            prior.position_covariance = np.diag([v, v, vz])
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
            n += 1
    print('robo rtk priors:', n, f'({len(DROP_NAMES)} dropped)')

In [ ]:
pairs = set()
for run, s in robo_sel.items():
    pairs |= sequential_pairs(s['names'], SEQ_K)
    for m in map_sel.values():
        pairs |= cross_pairs(s, m, CROSS_K, CROSS_R, HEADING_MAX_DEG)
for a, b in combinations(sorted(robo_sel), 2):
    pairs |= cross_pairs(robo_sel[a], robo_sel[b], CROSS_K, CROSS_R, HEADING_MAX_DEG)

(rtk_work / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(len(pairs), 'pairs')
match_pairs(rtk_db, rtk_work / 'pairs.txt')

# checkpoint the matched db (features + matches + priors) for BA experiments
ck = rtk_out / 'rtk_ckpt'
ck.mkdir(parents=True, exist_ok=True)
shutil.copy(rtk_db, ck / 'database.db')
with open(ck / 'robo_sel.json', 'w') as f:
    json.dump({run: {'names': s['names'], 'ts': [int(t) for t in s['ts']],
                     'enu': s['enu'].tolist(), 'heading': s['heading'].tolist()}
               for run, s in robo_sel.items()}, f)
shutil.copy(rtk_work / 'pairs.txt', ck / 'pairs.txt')

In [ ]:
# final pipeline: (1) register robo on the frozen map — priors OFF so the CASPAR GPU
# backend can run, the map itself anchors registration; (2) short direct prior BA pulls
# the field toward the anchors; (3) smooth prewarp of all frames onto the anchors, then
# FULL retriangulation (a nonrigid warp breaks rigid geometry — structure must be
# rebuilt, never kept); (4) thin the weakest tracks; (5) converged prior BA — from this
# basin it converges in a handful of iterations
base_opt = {'ba_use_gpu': True, 'ba_refine_sensor_from_rig': False,
            'ba_local_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
            'ba_global_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
            'ba_global_frames_ratio': 1.3, 'ba_global_points_ratio': 1.3,
            'ba_global_max_num_iterations': 30, 'ba_global_max_refinements': 2,
            'multiple_models': False, 'fix_existing_frames': True,
            'mapper': {'ba_global_ignore_redundant_points3D': True, 'max_reg_trials': 2}}

rtk_out.mkdir(parents=True, exist_ok=True)
p1 = rtk_out / 'phase1'
p1.mkdir(exist_ok=True)
with pycolmap.ostream():
    recs = pycolmap.incremental_mapping(rtk_db, IMAGES_PATH, p1, options=base_opt,
                                        input_path=str(rtk_map_model))
rtk_model = recs[0]
reg1 = {rtk_model.images[i].name for i in rtk_model.reg_image_ids()}
for run, s in robo_sel.items():
    print(f"{run}: registered {len(reg1 & set(s['names']))}/{len(s['names'])}")


def prior_ba(model, iterations, tol=None):
    """Direct pose-prior BA (bypasses the mapping pipeline). Robo priors scaled to
    sigma/PRIOR_SCALE; intrinsics frozen (calibrated in phase 1 — freeing them adds a
    scale/z drift mode). Priors with covariance ignore prior_position_fallback_stddev."""
    with pycolmap.Database.open(str(rtk_db)) as db:
        priors = []
        for p in db.read_all_pose_priors():
            i = p.corr_data_id.id
            if not (model.exists_image(i) and model.images[i].has_pose):
                continue
            if not model.images[i].name.split('/')[0].isdigit():
                p.position_covariance = p.position_covariance / PRIOR_SCALE**2
            priors.append(p)
    cfg = pycolmap.BundleAdjustmentConfig()
    for i in model.reg_image_ids():
        cfg.add_image(i)
    bo = pycolmap.BundleAdjustmentOptions()
    bo.refine_focal_length = False
    bo.refine_extra_params = False
    bo.refine_principal_point = False
    so = bo.ceres.solver_options
    so.max_num_iterations = iterations
    so.num_threads = -1
    if tol is not None:
        so.function_tolerance = tol
        so.use_inner_iterations = True
    pycolmap.create_pose_prior_ceres_bundle_adjuster(
        bo, pycolmap.PosePriorBundleAdjustmentOptions(), cfg, priors, model).solve()


prior_ba(rtk_model, 40)

# prewarp: long-wavelength displacement field (camera -> prior), applied to every frame
# helps converge more cleanly
from scipy.interpolate import RBFInterpolator

with pycolmap.Database.open(str(rtk_db)) as db:
    pri_pos = {p.corr_data_id.id: np.array(p.position) for p in db.read_all_pose_priors()}
robo_left = {n for s in robo_sel.values() for n in s['names']}
aids = [i for i in rtk_model.reg_image_ids()
        if i in pri_pos and rtk_model.images[i].name in robo_left]
P = np.array([rtk_model.images[i].projection_center() for i in aids])
D = np.array([pri_pos[i] - rtk_model.images[i].projection_center() for i in aids])
rbf = RBFInterpolator(P[:, :2], D, kernel='thin_plate_spline', smoothing=5000.0,
                      neighbors=256)
cap = np.percentile(np.linalg.norm(D, axis=1), 99)


def warp(xyz):
    d = rbf(xyz[:, :2])
    nn = np.linalg.norm(d, axis=1, keepdims=True)
    return xyz + d * np.minimum(1.0, cap / np.maximum(nn, 1e-9))


for f in rtk_model.frames.values():
    if not f.has_pose():
        continue
    R = f.rig_from_world.rotation
    c = -(R.matrix().T @ f.rig_from_world.translation)
    r2 = pycolmap.Rigid3d()
    r2.rotation = R
    r2.translation = -(R.matrix() @ warp(c[None])[0])
    f.rig_from_world = r2
for pid in list(rtk_model.points3D.keys()):
    rtk_model.delete_point3D(pid)
rtk_model = pycolmap.triangulate_points(rtk_model, str(rtk_db), IMAGES_PATH,
                                        str(rtk_out / 'pw_tri'))

# thin: drop track-2 points and subsample track<=4 — the discarded observations carry
# the least geometry and the most correlated error (softens the misspecified term)
rng = np.random.default_rng(0)
drop = [pid for pid, p in rtk_model.points3D.items()
        if p.track.length() <= 2 or (p.track.length() <= 4 and rng.random() > 0.6)]
for pid in drop:
    rtk_model.delete_point3D(pid)
print(f'thinned {len(drop)} points, {len(rtk_model.points3D)} remain')

prior_ba(rtk_model, 400, tol=1e-5)

In [ ]:
rtk_out.mkdir(parents=True, exist_ok=True)
(rtk_out / 'prior_all_rtk').mkdir(exist_ok=True)
rtk_model.write(rtk_out / 'prior_all_rtk')

# covariance-aware residuals vs the UNSCALED priors — the honest metrics; a raw z rmse
# vs RTK is misleading (per-day RTK z biases), judge vertical against the LiDAR DEM
with pycolmap.Database.open(str(rtk_db)) as db:
    priors = {p.corr_data_id.id: p for p in db.read_all_pose_priors()}
robo_left = {n for s in robo_sel.values() for n in s['names']}
rows = [(rtk_model.images[i].projection_center() - priors[i].position,
         float(np.sqrt(priors[i].position_covariance[0, 0])))
        for i in rtk_model.reg_image_ids()
        if i in priors and rtk_model.images[i].name in robo_left]
if rows:
    e = np.array([r[0] for r in rows])
    sig = np.array([r[1] for r in rows])
    h = np.hypot(e[:, 0], e[:, 1])
    norm = h / sig
    iw = np.sqrt((h**2 / sig**2).sum() / (1 / sig**2).sum())
    print(f'rtk anchors n={len(rows)}: info-weighted {iw:.3f}m | norm p50 '
          f'{np.median(norm):.1f} (nominal ~1.2) | within 1/2/3 sigma '
          f'{np.mean(norm<1):.0%}/{np.mean(norm<2):.0%}/{np.mean(norm<3):.0%} | '
          f'z rmse {np.sqrt((e[:, 2]**2).mean()):.2f}m (see DEM caveat)')
    for lo, hi, tag in [(0, 0.12, 'FIXED'), (0.12, 0.35, 'mid'), (0.35, 9, 'FLOAT')]:
        msk = (sig >= lo) & (sig < hi)
        if msk.sum():
            print(f'  {tag}: n={msk.sum()} rmse {np.sqrt((h[msk]**2).mean()):.3f}m '
                  f'norm p50 {np.median(norm[msk]):.1f}')
else:
    print('rtk prior residuals: no registered robo frames')

# per-VIRB-run agreement with (offset-corrected) racebox after anchoring — racebox is a
# sigma~2-3m instrument: this only guards against multi-meter blunders
for run in sorted(data, key=int):
    ims = [(int(Path(rtk_model.images[i].name).stem), rtk_model.images[i].projection_center())
           for i in rtk_model.reg_image_ids()
           if rtk_model.images[i].name.split('/')[0] == run]
    if not ims:
        continue
    fts = np.array([t for t, _ in ims], float)
    pos = np.array([p for _, p in ims])
    rts, renu = virb_enu[run]
    r = np.stack([np.interp(fts, rts, renu[:, k]) for k in range(2)], 1)
    e = pos[:, :2] - r
    print(f'{run}: vs racebox {np.sqrt((np.hypot(*e.T)**2).mean()):.2f}m rmse, '
          f'mean ({e[:, 0].mean():+.2f},{e[:, 1].mean():+.2f})')

# Visualize

### No Prior

In [13]:
# model = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'no_prior'))

In [15]:
# fig = viz_3d.init_figure()
# viz_3d.plot_reconstruction(fig, model, points_rgb=True)
# fig.show()

### Prior

In [ ]:
model_prior = pycolmap.Reconstruction(str(outputs / 'prior'))

In [ ]:
with pycolmap.Database.open(str(database)) as db:
    priors = db.read_all_pose_priors()
enu = {p.corr_data_id.id: p.position for p in priors}  # already ENU
errs = np.array([model_prior.images[i].projection_center() - enu[i]
                 for i in model_prior.reg_image_ids()])
print(f"rmse vs gps: {np.sqrt((errs[:, :2] ** 2).sum(1).mean()):.2f} m horizontal, "
      f"{np.sqrt((errs[:, 2] ** 2).mean()):.2f} m vertical")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

by_run = {}
for i in sorted(model_prior.reg_image_ids()):
    by_run.setdefault(model_prior.images[i].name.split('/')[0], []).append(i)

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(by_run, key=int)):
    ids = by_run[run]
    pri = np.array([enu[i] for i in ids])
    sol = np.array([model_prior.images[i].projection_center() for i in ids])
    names = [model_prior.images[i].name for i in ids]
    c = colors[k % len(colors)]
    seg = np.concatenate([pri[:, None, :2], sol[:, None, :2],
                          np.full((len(ids), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=pri[:, 0], y=pri[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=4),
                    name=f'{run} prior', legendgroup=run)
    fig.add_scatter(x=sol[:, 0], y=sol[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} solved', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model_prior, points_rgb=True)
fig.show()

# Localization

Register query runs against the fixed `prior_all` model, one run at a time. Runs on copies — the original database/model are never modified — and query frames only match the map (no pairs between localized runs). Runnable standalone: Setup → Config → here.

In [ ]:
# TODO do against better model
map_database = DATA_PATH / 'database.db'
map_model_path = DATA_PATH / 'prior_all'
loc_work = Path('/tmp/loc')  # db copies + scratch
loc_out = Path('/kaggle/working/loc')

LOC_RUNS = None        # subset of int run ids; None = all of test_vid_imu
LOC_DELTA_S = 2.0      # m between query frames
LOC_GPS = 'racebox'    # 'racebox' | 'virb': selection + pairing source (virb skips uncovered runs)
LOC_SEQ_K = SEQ_K
LOC_CROSS_K = CROSS_K  # map candidates per query frame, per map run
LOC_CROSS_R = CROSS_R
LOC_PRIORS = False     # virb position priors on query frames
LOC_PRIOR_STD_XY, LOC_PRIOR_STD_Z = 3.0, 5.0

In [ ]:
loc_videos = {v.stem: v for v in (DATA_PATH / 'test_vid').glob('*.[mM][pP]4')}
loc_data = load_vid_imu(DATA_PATH / 'test_vid_imu')
loc_runs = [r for r in sorted(loc_data, key=int)
            if LOC_RUNS is None or int(r) in LOC_RUNS]
if LOC_GPS == 'virb':
    skipped = [r for r in loc_runs if not loc_data[r]['gps_data']]
    loc_runs = [r for r in loc_runs if loc_data[r]['gps_data']]
    if skipped:
        print('no virb gps, skipping:', skipped)

for run in tqdm(loc_runs, desc='videos'):
    extract_frames(loc_videos[run], loc_data[run]['camera_start'], IMAGES_PATH / run)
loc_runs

In [ ]:
loc_sel = {}
for run in loc_runs:
    loc_sel[run], _ = select_frames(run, loc_data[run], LOC_GPS, LOC_DELTA_S)

# map frame positions/headings come from the model itself, not the mapping vid_imu
map_model = pycolmap.Reconstruction(str(map_model_path))
by_run = defaultdict(list)
for i in map_model.reg_image_ids():
    im = map_model.images[i]
    p = Path(im.name)
    by_run[p.parts[0]].append((int(p.stem), im.name, im.projection_center()))
map_sel = {}
for run, items in by_run.items():
    items.sort()
    enu = np.array([c for _, _, c in items])
    grad = np.gradient(enu[:, :2], axis=0)
    map_sel[run] = {'names': [n for _, n, _ in items], 'enu': enu,
                    'heading': np.arctan2(grad[:, 1], grad[:, 0])}
print({run: len(s['names']) for run, s in map_sel.items()})

In [ ]:
stage_models()
link_masks([n for run in loc_runs for n in loc_sel[run]['names']])
loc_work.mkdir(parents=True, exist_ok=True)
local_map_db = loc_work / 'map_database.db'
if not local_map_db.exists():
    shutil.copy(map_database, local_map_db)  # Drive reads are slow; stage the 4.4 GB db once

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, loc_work / 'colmap.LOG.')
loc_models = {}
for run in loc_runs:
    names = loc_sel[run]['names']
    work = loc_work / run
    db = work / 'database.db'
    if db.exists():  # reuse features/matches only if the selection is unchanged
        with pycolmap.Database.open(str(db)) as dbh:
            same = {im.name for im in dbh.read_all_images()
                    if im.name.split('/')[0] == run} == set(names)
        if not same:
            shutil.rmtree(work)
    if not db.exists():
        work.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_map_db, db)
        extract_image_features(db, names)

    pairs = sequential_pairs(names, LOC_SEQ_K)
    for m in map_sel.values():
        pairs |= cross_pairs(loc_sel[run], m, LOC_CROSS_K, LOC_CROSS_R, HEADING_MAX_DEG)
    (work / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
    match_pairs(db, work / 'pairs.txt')

    if LOC_PRIORS:
        write_pose_priors(db, {run: gps_enu(loc_data[run], 'gps_data')},
                          np.diag([LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_Z**2]))

    # priors only constrain global BA, and caspar doesn't support them
    opt = {'fix_existing_frames': True, 'use_prior_position': LOC_PRIORS,
           'ba_use_gpu': True,
           'ba_local_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
           'ba_global_backend': (pycolmap.BundleAdjustmentBackend.CERES if LOC_PRIORS
                                 else pycolmap.BundleAdjustmentBackend.CASPAR)}

    shutil.rmtree(work / 'model', ignore_errors=True)
    (work / 'model').mkdir()
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(db, IMAGES_PATH, work / 'model',
                                            options=opt, input_path=str(map_model_path))
    loc_models[run] = recs[0]
    reg = {loc_models[run].images[i].name for i in loc_models[run].reg_image_ids()}
    print(f'{run}: registered {len(reg & set(names))}/{len(names)} query frames')

In [ ]:
loc_results = {}
for run, model in loc_models.items():
    names = set(loc_sel[run]['names'])
    rb_ts, rb_enu = gps_enu(loc_data[run], 'racebox_gps')
    frames = []
    for i in sorted(model.reg_image_ids()):
        im = model.images[i]
        if im.name not in names:
            continue
        ts = int(Path(im.name).stem)
        c = im.projection_center()
        rb = np.array([np.interp(ts, rb_ts, rb_enu[:, j]) for j in range(3)])
        frames.append({'ts': ts, 'enu': c.tolist(),
                       'quat_xyzw': im.cam_from_world().rotation.quat.tolist(),
                       'racebox_err': (c - rb).tolist()})
    loc_results[run] = frames

    out = loc_out / run
    out.mkdir(parents=True, exist_ok=True)
    model.write(out)
    with open(out / 'poses.json', 'w') as f:
        json.dump({'delta_s': LOC_DELTA_S, 'gps': LOC_GPS, 'priors': LOC_PRIORS,
                   'frames': frames}, f)

    errs = np.array([f['racebox_err'] for f in frames])
    h = np.hypot(errs[:, 0], errs[:, 1])
    print(f"{run}: {len(frames)}/{len(names)} frames, vs racebox "
          f"{np.sqrt((h**2).mean()):.2f}m rmse / {np.percentile(h, 90):.2f}m p90 horizontal, "
          f"{np.sqrt((errs[:, 2]**2).mean()):.2f}m rmse vertical")

In [ ]:
# reload saved localizations (skip the reconstruction cells above)
# import json
# from pathlib import Path
# loc_out = Path('../../../.././tmp/loc_out')
loc_models, loc_results = {}, {}
for p in sorted(loc_out.iterdir()):
    if (p / 'poses.json').exists():
        with open(p / 'poses.json') as f:
            loc_results[p.name] = json.load(f)['frames']
        loc_models[p.name] = pycolmap.Reconstruction(str(p))
print(sorted(loc_results))

['39']


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import numpy as np
# pio.renderers.default = "notebook_connected"

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = loc_results[run]
    enu = np.array([f['enu'] for f in frames])
    errs = np.array([f['racebox_err'] for f in frames])
    rb = enu - errs
    text = [f'{run}/{f["ts"]}: {np.hypot(*e[:2]):.2f}m' for f, e in zip(frames, errs)]
    c = colors[k % len(colors)]
    seg = np.concatenate([rb[:, None, :2], enu[:, None, :2],
                          np.full((len(frames), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=rb[:, 0], y=rb[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=4),
                    name=f'{run} racebox', legendgroup=run)
    fig.add_scatter(x=enu[:, 0], y=enu[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} loc', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

fig = go.Figure()
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = sorted(loc_results[run], key=lambda f: f['ts'])
    ts = np.array([f['ts'] for f in frames]) / 1e9
    h = np.array([np.hypot(*f['racebox_err'][:2]) for f in frames])
    fig.add_scatter(x=ts - ts[0], y=h, mode='markers', text=[f['ts'] for f in frames],
                    marker=dict(color=colors[k % len(colors)], size=4), name=run)
fig.update_layout(height=350, xaxis_title='s', yaxis_title='horizontal error m')
fig.show()

## PnP

Per-frame localization against the fixed `rtk_base` map: match query frames to map images,
lift matches to 2D-3D through the model, solve every frame independently (LO-RANSAC +
refinement). Matching happens once per run as a generous **superset**; every narrower
strategy (map-run subset, per-run k, gating radius, match budget, gating source) is a local
filter over the matched pairs — no re-matching. Query focal length must be estimated per
run (stabilizer state shifts it ~20% and PnP absorbs focal error as an along-view position
bias): pass 1 refines focal per frame, pass 2 re-solves with the run median fixed.

Colab side: superset matching cell (chunked, checkpoints a compact query-side delta after
each chunk — survives VM reclaim; features are dumped once so keypoint indexing is stable
across sessions). Local side: delta + `rtk_base/model` are enough for the whole sweep.

In [ ]:
PNP_MAP = DATA_PATH / 'rtk_base'
pnp_work = Path('/tmp/pnp')
pnp_out = Path('/kaggle/working/pnp') if Path('/kaggle').exists() else outputs / 'pnp'

PNP_SEQ_K = 6            # sequential pairs (kept for joint-BA experiments; PnP skips them)
PNP_K_VIRB, PNP_K_ROBO = 4, 3
PNP_CROSS_R, PNP_HEAD_MAX = 12.0, 55
PNP_CHUNK = 5000         # pairs per matching chunk; delta checkpoint after each
PNP_CKPT = None          # rclone remote dir for checkpoints, e.g. 'drive:buggy_hloc/pnp'
PNP_MIN_CORR = 12

In [ ]:
MAXI = 2147483647


def obj_array(arrs):
    # np.array(arrs, dtype=object) collapses to N-D whenever the arrays share a shape
    out = np.empty(len(arrs), dtype=object)
    for i, a in enumerate(arrs):
        out[i] = a
    return out


def export_pnp_delta(db_path, sel, out_npz):
    """Query keypoints + camera + verified matches touching query images, as one npz."""
    con = sqlite3.connect(db_path)
    name_to_id = {n: i for i, n in con.execute('select image_id, name from images')}
    id_to_name = {i: n for n, i in name_to_id.items()}
    qids = {name_to_id[n] for n in sel['names'] if n in name_to_id}
    kps = {i: np.frombuffer(r[2], np.float32).reshape(r[0], r[1])
           for i in qids
           for r in con.execute('select rows, cols, data from keypoints where image_id=?', (i,))}
    qcams = {con.execute('select camera_id from images where image_id=?', (i,)).fetchone()[0]
             for i in qids}
    cams = {c: (r[0], r[1], r[2], np.frombuffer(r[3], np.float64))
            for c in qcams
            for r in con.execute('select model, width, height, params from cameras'
                                 ' where camera_id=?', (c,))}
    tvg = {}
    for pid, rows, cols, data, config in con.execute(
            'select pair_id, rows, cols, data, config from two_view_geometries where rows > 0'):
        i1, i2 = pid // MAXI, pid % MAXI
        if i1 in qids or i2 in qids:
            tvg[pid] = (i1, i2, config,
                        np.frombuffer(data, np.uint32).reshape(rows, cols) if data else
                        np.zeros((0, 2), np.uint32))
    con.close()
    np.savez_compressed(
        out_npz,
        sel_names=np.array(sel['names']), sel_enu=sel['enu'], sel_heading=sel['heading'],
        image_names_all=np.array([id_to_name[i] for i in sorted(id_to_name)]),
        image_ids_all=np.array(sorted(id_to_name)),
        kp_ids=np.array(sorted(kps)), kp_data=obj_array([kps[i] for i in sorted(kps)]),
        cam_ids=np.array(sorted(cams)),
        cam_data=np.array([(cams[c][0], cams[c][1], cams[c][2], *cams[c][3])
                           for c in sorted(cams)]),
        tvg_pairs=np.array([(v[0], v[1], v[2]) for v in tvg.values()]),
        tvg_matches=obj_array([v[3] for v in tvg.values()]),
        allow_pickle=True)
    return len(tvg)

In [ ]:
# superset matching per run (GPU). Reuses loc_sel/map_sel from the cells above.
pnp_work.mkdir(parents=True, exist_ok=True)
pnp_out.mkdir(parents=True, exist_ok=True)
ROBO_RUNS = {r for r in map_sel if not r.split('/')[0].isdigit()}

for run in loc_runs:
    sel = loc_sel[run]
    db = pnp_work / f'{run}.db'
    if not db.exists():
        shutil.copy(PNP_MAP / 'database.db', db)
        extract_image_features(db, sel['names'])
    pairs = sequential_pairs(sel['names'], PNP_SEQ_K)
    for mrun, ms in map_sel.items():
        k = PNP_K_ROBO if mrun in ROBO_RUNS else PNP_K_VIRB
        pairs |= cross_pairs(sel, ms, k, PNP_CROSS_R, PNP_HEAD_MAX)
    con = sqlite3.connect(db)
    name_to_id = {n: i for i, n in con.execute('select image_id, name from images')}
    qids = {name_to_id[n] for n in sel['names']}
    done = set()
    for pid, in con.execute('select pair_id from two_view_geometries'):
        if pid // MAXI in qids or pid % MAXI in qids:
            done.add(pid)
    con.close()

    def pid_of(a, b):
        ia, ib = sorted((name_to_id[a], name_to_id[b]))
        return ia * MAXI + ib

    todo = sorted(p for p in pairs if pid_of(*p) not in done)
    print(f'{run}: {len(pairs)} pairs, {len(todo)} to match')
    delta = pnp_out / f'delta_{run}.npz'
    for c0 in range(0, len(todo), PNP_CHUNK):
        pf = pnp_work / 'chunk.txt'
        pf.write_text('\n'.join(f'{a} {b}' for a, b in todo[c0:c0 + PNP_CHUNK]))
        match_pairs(db, pf)
        n = export_pnp_delta(db, sel, delta)
        if PNP_CKPT:
            subprocess.run(['rclone', 'copyto', str(delta), f'{PNP_CKPT}/delta_{run}.npz'])
        print(f'{run}: chunk {c0 // PNP_CHUNK + 1}/{-(-len(todo) // PNP_CHUNK)}, '
              f'{n} matched pairs checkpointed')

In [ ]:
# local: correspondence groups from delta(s) + model; strategies filter these
pnp_model = pycolmap.Reconstruction(str(PNP_MAP / 'model'))
pnp_xyz = {pid: p.xyz for pid, p in pnp_model.points3D.items()}
pnp_mid = {pnp_model.images[i].name: i for i in pnp_model.reg_image_ids()}
_p3d_cache = {}


def _map_p3d(name):
    if name not in _p3d_cache:
        mid = pnp_mid.get(name)
        if mid is None:
            _p3d_cache[name] = None
        else:
            im = pnp_model.images[mid]
            arr = np.full(len(im.points2D), -1, np.int64)
            for k, p2 in enumerate(im.points2D):
                if p2.has_point3D():
                    arr[k] = p2.point3D_id
            _p3d_cache[name] = (arr, im.projection_center())
    return _p3d_cache[name]


def load_pnp_groups(delta_paths):
    """{query name: [group]}, each group one matched map image with its 2D-3D corrs."""
    kp_by_name, sel, cam_row, seen = {}, None, None, set()
    groups = defaultdict(list)
    for dp in delta_paths:
        d = np.load(dp, allow_pickle=True)
        name_of = dict(zip(d['image_ids_all'].tolist(), d['image_names_all'].tolist()))
        for i, k in zip(d['kp_ids'].tolist(), d['kp_data']):
            kp_by_name[name_of[i]] = k
        if sel is None or len(d['sel_names']) > len(sel['names']):
            sel = {'names': list(d['sel_names']), 'enu': d['sel_enu'],
                   'heading': d['sel_heading']}
        if cam_row is None:
            cam_row = next(iter(d['cam_data']))
        qnames = set(d['sel_names'])
        for (i1, i2, cfg), matches in zip(d['tvg_pairs'], d['tvg_matches']):
            n1, n2 = name_of[int(i1)], name_of[int(i2)]
            q, mp = (n1, n2) if n1 in qnames else (n2, n1) if n2 in qnames else (None, None)
            if q is None or mp in qnames or (q, mp) in seen or not len(matches):
                continue
            seen.add((q, mp))
            got = _map_p3d(mp)
            if got is None:
                continue
            p3, mc = got
            qm = matches if n1 == q else matches[:, ::-1]
            valid = p3[qm[:, 1]] >= 0
            if not valid.any():
                continue
            groups[q].append({'mrun': mp.split('/')[0], 'mcenter': mc[:2],
                              'q_xy': kp_by_name[q][qm[valid, 0]][:, :2],
                              'pts': np.array([pnp_xyz[p3[j]] for j in qm[valid, 1]])})
    return groups, sel, cam_row


def pnp_solve(groups, sel, cam_row, cfg, stride=1):
    """cfg: runs, k, r, max_pairs, min_corr, focal ('auto'|value|None), refine_focal,
    gate positions default to sel['enu'] (racebox); pass gate_pos to override."""
    gpos = cfg.get('gate_pos', np.asarray(sel['enu'])[:, :2])
    ref = pycolmap.AbsolutePoseRefinementOptions()
    est = pycolmap.AbsolutePoseEstimationOptions()
    if cfg.get('refine_focal'):
        ref.refine_focal_length = ref.refine_extra_params = True
    cam_tpl = pycolmap.Camera(
        camera_id=1, model=pycolmap.CameraModelId(int(cam_row[0])),
        width=int(cam_row[1]), height=int(cam_row[2]), params=list(cam_row[3:]))
    if cfg.get('focal') == 'auto':
        probe = dict(cfg, focal=None, refine_focal=True)
        rows = pnp_solve(groups, sel, cam_row, probe, stride=4)
        cfg = dict(cfg, focal=float(np.median([r['focal'] for r in rows if r])),
                   k1=float(np.median([r['k1'] for r in rows if r])),
                   refine_focal=False)
        print(f"auto focal -> {cfg['focal']:.1f} k1 -> {cfg['k1']:+.4f}")
    rows = []
    for i, qname in enumerate(sel['names']):
        if i % stride:
            rows.append(None)
            continue
        gs = groups.get(qname, [])
        if cfg.get('runs') is not None:
            gs = [g for g in gs if g['mrun'] in cfg['runs']]
        ds = [float(np.hypot(*(g['mcenter'] - gpos[i]))) for g in gs]
        if cfg.get('r') is not None:
            gs, ds = zip(*[(g, dd) for g, dd in zip(gs, ds) if dd <= cfg['r']]) \
                if any(dd <= cfg['r'] for dd in ds) else ([], [])
        if cfg.get('k') is not None:
            byrun = defaultdict(list)
            for g, dd in zip(gs, ds):
                byrun[g['mrun']].append((dd, g))
            gs = [g for lst in byrun.values()
                  for _, g in sorted(lst, key=lambda x: x[0])[:cfg['k']]]
        elif cfg.get('max_pairs') is not None:
            gs = [gs[j] for j in np.argsort(ds)[:cfg['max_pairs']]]
        if not gs:
            rows.append(None)
            continue
        q_xy = np.vstack([g['q_xy'] for g in gs])
        pts = np.vstack([g['pts'] for g in gs])
        if len(q_xy) < cfg.get('min_corr', PNP_MIN_CORR):
            rows.append(None)
            continue
        camera = pycolmap.Camera(camera_id=1, model=cam_tpl.model, width=cam_tpl.width,
                                 height=cam_tpl.height, params=list(cam_tpl.params))
        if isinstance(cfg.get('focal'), float):
            camera.params = [cfg['focal'], *camera.params[1:]]
        if cfg.get('k1') is not None:
            camera.params = [*camera.params[:3], cfg['k1']]
        r = pycolmap.estimate_and_refine_absolute_pose(q_xy, pts, camera, est, ref)
        rows.append(None if r is None else
                    {'center': r['cam_from_world'].inverse().translation,
                     'inl': int(np.count_nonzero(r['inlier_mask'])),
                     'focal': float(camera.params[0]),
                     'k1': float(camera.params[3]), 'npairs': len(gs)})
    return rows

In [ ]:
def pnp_metrics(rows, sel, gps_ts, gps_enu_arr, label=''):
    """dt-refit vs the gps series, then residual stats + consecutive-frame jitter."""
    ok = [(i, r) for i, r in enumerate(rows) if r]
    idx = np.array([i for i, _ in ok])
    C = np.array([r['center'] for _, r in ok])
    ts_q = np.array([int(sel['names'][i].split('/')[1].split('.')[0]) for i in idx], float)
    best = min(((dt, float(np.median(np.hypot(
        *(C[:, :2] - np.stack([np.interp(ts_q + dt * 1e6, gps_ts, gps_enu_arr[:, j])
                               for j in range(2)], 1)).T))))
        for dt in np.arange(-1500, 1501, 10.0)), key=lambda x: x[1])
    dt = best[0]
    g = np.stack([np.interp(ts_q + dt * 1e6, gps_ts, gps_enu_arr[:, j]) for j in range(3)], 1)
    e = C - g
    h = np.hypot(e[:, 0], e[:, 1])
    dv = np.linalg.norm(np.diff(C[:, :2], axis=0) - np.diff(g[:, :2], axis=0), axis=1)
    jit = dv[np.diff(idx) == 1]
    out = {'loc': len(ok), 'total': len(rows), 'dt_ms': dt,
           'p50': float(np.median(h)), 'p90': float(np.percentile(h, 90)),
           'rmse': float(np.sqrt((h ** 2).mean())),
           'z_mean': float(e[:, 2].mean()),
           'inl_p50': int(np.median([r['inl'] for _, r in ok])),
           'jit_p50': float(np.median(jit)), 'jit_p90': float(np.percentile(jit, 90))}
    print(f"{label:26} loc {out['loc']}/{out['total']} dt {dt:+5.0f}ms | "
          f"p50 {out['p50']:.3f} p90 {out['p90']:.3f} rmse {out['rmse']:.3f} | "
          f"z {out['z_mean']:+.2f} inl {out['inl_p50']} | "
          f"jit {out['jit_p50']:.3f}/{out['jit_p90']:.3f}")
    return out

In [ ]:
VIRB_MAP_RUNS = sorted(r for r in map_sel if r.split('/')[0].isdigit())
ROBO_MAP_RUNS = sorted(r for r in map_sel if not r.split('/')[0].isdigit())
PNP_STRATEGIES = {
    'virb_k2_r6': {'runs': VIRB_MAP_RUNS, 'k': 2, 'r': 6.0, 'focal': 'auto'},
    'virb_k4_r12': {'runs': VIRB_MAP_RUNS, 'k': 4, 'r': 12.0, 'focal': 'auto'},
    'all_k2_r6': {'k': 2, 'r': 6.0, 'focal': 'auto'},
    'all_k4_r12': {'k': 4, 'r': 12.0, 'focal': 'auto'},
    'robo_k3_r12': {'runs': ROBO_MAP_RUNS, 'k': 3, 'r': 12.0, 'focal': 'auto'},
    'budget4': {'max_pairs': 4, 'r': 12.0, 'focal': 'auto'},
    'budget16': {'max_pairs': 16, 'r': 12.0, 'focal': 'auto'},
}

pnp_results = {}
for run in loc_runs:
    deltas = sorted(pnp_out.glob(f'delta*{run}*.npz'))
    if not deltas:
        continue
    groups, sel, cam_row = load_pnp_groups(deltas)
    gps_ts, gps_enu_arr = gps_enu(loc_data[run], 'racebox_gps')
    print(f'=== {run}: {sum(len(v) for v in groups.values())} matched pair-groups ===')
    for name, cfg in PNP_STRATEGIES.items():
        rows = pnp_solve(groups, sel, cam_row, dict(cfg))
        pnp_results[(run, name)] = (rows, pnp_metrics(rows, sel, gps_ts, gps_enu_arr, name))

In [ ]:
# review overlay: gps track vs per-strategy PnP centers, colored by inlier count
import plotly.graph_objects as go

run = loc_runs[0]
fig = go.Figure()
gps_ts, gps_enu_arr = gps_enu(loc_data[run], 'racebox_gps')
fig.add_trace(go.Scatter(x=gps_enu_arr[:, 0], y=gps_enu_arr[:, 1], mode='lines',
                         name='racebox', line=dict(color='gray', width=1)))
for name in PNP_STRATEGIES:
    rows, _ = pnp_results.get((run, name), (None, None))
    if not rows:
        continue
    C = np.array([r['center'] for r in rows if r])
    inl = [r['inl'] for r in rows if r]
    fig.add_trace(go.Scatter(x=C[:, 0], y=C[:, 1], mode='markers', name=name,
                             marker=dict(size=4, color=inl, colorscale='Viridis'),
                             visible='legendonly' if name != 'all_k2_r6' else True))
fig.update_layout(height=700, yaxis_scaleanchor='x')
fig.show()